In [ ]:
from torch.utils.data import DataLoader
import numpy as np
import os
import pandas as pd
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

from core.Log import *
from core.plots import *
from core.modelUtils import *
from core.CNNmodel import *
from core.CardiacCTdataset import *
from core.preprocessing import *

import logging

folds_logger()
root_logger()
logger = logging.getLogger('root')

CNTRL_dicom_root = "../Takotsubo-Syndrome/data/Inputs/normal_cases/"
TTS_dicom_root = "../Takotsubo-Syndrome/data/Inputs/takotsubo_cases/"
root_dir = "data/cases/"



In [ ]:
full_datalist = load_dataset_info()
full_indices = np.arange(len(full_datalist))
full_labels = [d['label'] for d in full_datalist]
tts_n = 0
cntrl_n = 0
for lbl in full_labels:
	if lbl == 1: tts_n += 1
	else: cntrl_n += 1
logger.info(f"Total Cases: {len(full_datalist)} | TTS: {tts_n}, Control: {cntrl_n}")

final_scores = []
epochs_history = []

hypers = {"LR": 1e-4, "WD": 1e-5, "epochs": 2,
			  "patience": 10, "batch_size": 4, "threshold_cutoff": 0.5, "DR": 0.4}

'''
Outer Split: 80% Train+Val Set,  20% Test Set
Inner Split: of 80% --> 60% Train Set, 20% Val Set
'''
N_OUTER_SPLITS = 3
outer_cv = StratifiedShuffleSplit(n_splits=N_OUTER_SPLITS, test_size=0.2, random_state=42)
logger.info(f"Starting {N_OUTER_SPLITS}-fold Nested Cross-Validation...")

for fold_idx, (train_val_idx, test_idx) in enumerate(outer_cv.split(full_indices, full_labels)):

	# --- 3. Outer Split: (Train+Val Pool) vs. Test Set ---
	test_datalist_fold = [full_datalist[i] for i in test_idx]
	train_val_datalist_fold = [full_datalist[i] for i in train_val_idx]
	train_val_labels_fold = [d['label'] for d in train_val_datalist_fold]

	# --- 4. Inner Split: Train Set vs. Validation Set ---
	inner_sss = StratifiedShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
	train_idx, val_idx = next(inner_sss.split(np.arange(len(train_val_datalist_fold)), train_val_labels_fold))
	train_datalist_fold = [train_val_datalist_fold[i] for i in train_idx]
	val_datalist_fold = [train_val_datalist_fold[i] for i in val_idx]

	logger.info(f"Starting Fold {fold_idx + 1} | Data Split: "
		f"{len(train_datalist_fold)} train, "
		f"{len(val_datalist_fold)} val, "
		f"{len(test_datalist_fold)} test.")

	fold_scores, epochs_hist = run_k_fold(
		train_datalist=train_datalist_fold,
		val_datalist=val_datalist_fold,
		test_datalist=test_datalist_fold,
		fold_idx=fold_idx)
	final_scores['fold_id'] = fold_idx + 1
	final_scores['model_name'] = 'MultiViewCNN'#

	logger.info(f"End of Fold {fold_idx + 1} | Test results ---> {fold_scores}")
	epochs_history.append(epochs_hist)
	final_scores.append(fold_scores)



In [ ]:


final_df = pd.DataFrame([final_scores])
final_df.to_csv('folds_final_scores.csv', mode='a', header=not os.path.exists('folds_final_scores.csv'), index=False)
epoch_df = pd.DataFrame(epochs_history)
epoch_df.to_csv('folds_E_history.csv', mode='a', header=not os.path.exists('folds_E_history.csv'), index=False)

auc_scores = [s['auc'] for s in final_scores]

mean_score = np.mean(auc_scores)
std_score = np.std(auc_scores)
logger.info("--- Nested Cross-Validation Complete ---")
logger.info(f"Final Scores across {N_OUTER_SPLITS} folds: {[f'{s:.4f}' for s in auc_scores]}")
logger.info(f"Average Model Performance: {mean_score:.4f} ± {std_score:.4f}")
